# Dataset Overview

This notebook summarizes the variant-labeling datasets produced by the NGS pipeline across first-round and second-round roots.
It reads summary CSVs where available and raw `*.variant_labeling.csv` files for roots that do not have a summary file yet.

Labels are counted from the summary columns: **specific** = `n_spec_1`, **unspecific** = `n_spec_0`, and **uncertain** = `n_spec_2`.

A3 Q20 and Q30 are available for comparison below; DMF5 is autodetected and plotted when present.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.patches import Patch
from IPython.display import display

base = Path("/cluster/project/reddy/katja/NGS_pipeline/data")

summary_filenames = ["variant_labeling_summary.csv", "script4_variant_labeling_summary.csv"]


def summarize_summary_file(summary_path, dataset, quality, mode_label, mode_folder):
    summary_df = pd.read_csv(summary_path)
    n_variants = pd.to_numeric(summary_df.get("n_variants"), errors="coerce").fillna(0)
    n_spec_0 = pd.to_numeric(summary_df.get("n_spec_0"), errors="coerce").fillna(0)
    n_spec_1 = pd.to_numeric(summary_df.get("n_spec_1"), errors="coerce").fillna(0)
    n_spec_2 = pd.to_numeric(summary_df.get("n_spec_2"), errors="coerce").fillna(0)

    total_labels = int(n_variants.sum())
    specific_labels = int(n_spec_1.sum())
    unspecific_labels = int(n_spec_0.sum())
    uncertain_labels = int(n_spec_2.sum())
    peptides = int(summary_df["peptide"].nunique()) if "peptide" in summary_df.columns else total_labels

    return {
        "dataset": dataset,
        "quality": quality,
        "mode_folder": mode_folder,
        "mode_label": mode_label,
        "peptides": peptides,
        "total_labels": total_labels,
        "uncertain_labels": uncertain_labels,
        "unspecific_labels": unspecific_labels,
        "specific_labels": specific_labels,
        "uncertain_share": uncertain_labels / total_labels if total_labels else 0,
        "unspecific_share": unspecific_labels / total_labels if total_labels else 0,
        "specific_share": specific_labels / total_labels if total_labels else 0,
        "summary_path": str(summary_path),
    }


def summarize_raw_root(root, dataset, quality, mode_label):
    raw_files = sorted(root.glob("*.variant_labeling.csv"))
    total_labels = 0
    specific_labels = 0
    unspecific_labels = 0
    uncertain_labels = 0
    peptides = set()

    for raw_path in raw_files:
        raw_df = pd.read_csv(raw_path)
        total_labels += len(raw_df)
        specificity = pd.to_numeric(raw_df.get("specificity"), errors="coerce").fillna(-1)
        specific_labels += int((specificity == 1).sum())
        unspecific_labels += int((specificity == 0).sum())
        uncertain_labels += int((specificity == 2).sum())
        if "peptide" in raw_df.columns:
            peptides.update(raw_df["peptide"].dropna().astype(str).unique())

    return {
        "dataset": dataset,
        "quality": quality,
        "mode_folder": root.name,
        "mode_label": mode_label,
        "peptides": len(peptides) if peptides else len(raw_files),
        "total_labels": total_labels,
        "uncertain_labels": uncertain_labels,
        "unspecific_labels": unspecific_labels,
        "specific_labels": specific_labels,
        "uncertain_share": uncertain_labels / total_labels if total_labels else 0,
        "unspecific_share": unspecific_labels / total_labels if total_labels else 0,
        "specific_share": specific_labels / total_labels if total_labels else 0,
        "summary_path": "raw variant_labeling files",
    }


def collect_from_summary_dirs(root, dataset, quality, mode_labels):
    rows = []
    if not root.exists():
        return rows

    for mode_dir in sorted(path for path in root.iterdir() if path.is_dir()):
        for summary_name in summary_filenames:
            summary_path = mode_dir / summary_name
            if summary_path.exists():
                rows.append(
                    summarize_summary_file(
                        summary_path,
                        dataset=dataset,
                        quality=quality,
                        mode_label=mode_labels.get(mode_dir.name, mode_dir.name),
                        mode_folder=mode_dir.name,
                    )
                )
                break

    return rows


dataset_specs = [
    {
        "dataset": "A3",
        "quality": "second",
        "root": base / "P3408_LUCA-TCRA3/04_variant_labeling/19_05_2026_minlenght191_Q30",
        "source": "summary_dirs",
        "mode_labels": {
            "combined": "combined",
            "monotonic": "monotonic",
            "single_positive_3x": "3x enriched",
            "single_positive_3x1x": "3x1x enriched",
        },
    },
    {
        "dataset": "A3",
        "quality": "first",
        "root": base / "P3408_LUCA-TCRA3/04_variant_labeling/07_01_2026_minlength191_Q20",
        "source": "raw_root",
        "mode_label": "combined",
    },
    {
        "dataset": "DMF5",
        "quality": "second",
        "root": base / "P3481_LUCA-TCRDMF5/04_variant_labeling",
        "source": "summary_dirs",
        "mode_labels": {
            "combined": "combined",
            "monotonic": "monotonic",
            "single_positive_pos1x": "1x enriched",
            "single_positive_pos2x": "2x enriched",
        },
    },
    ]

pending_rows = [
    {"dataset": "A3", "quality": "first-quality", "status": "not yet produced", "expected_modes": "3x enriched, 3x1x enriched, combined, monotonic"},
    {"dataset": "DMF5", "quality": "first-quality", "status": "not yet produced", "expected_modes": "1x enriched, 2x1x enriched, combined, monotonic"},
]

overview_rows = []
source_rows = 0

for spec in dataset_specs:
    root = spec["root"]
    if not root.exists():
        continue

    if spec["source"] == "summary_dirs":
        rows = collect_from_summary_dirs(root, spec["dataset"], spec["quality"], spec.get("mode_labels", {}))
        overview_rows.extend(rows)
        source_rows += sum(row["total_labels"] for row in rows)
        continue

    if spec["source"] == "summary_file":
        summary_path = spec["root"] / spec["summary_file"]
        if summary_path.exists():
            row = summarize_summary_file(
                summary_path,
                dataset=spec["dataset"],
                quality=spec["quality"],
                mode_label=spec.get("mode_label", spec["root"].name),
                mode_folder=spec["root"].name,
            )
            overview_rows.append(row)
            source_rows += row["total_labels"]
        continue

    if spec["source"] == "raw_root":
        row = summarize_raw_root(
            root,
            dataset=spec["dataset"],
            quality=spec["quality"],
            mode_label=spec.get("mode_label", root.name),
        )
        overview_rows.append(row)
        source_rows += row["total_labels"]
        continue

overview = pd.DataFrame(overview_rows).sort_values(["dataset", "quality", "mode_label"]).reset_index(drop=True)
overview = overview[[
    "dataset",
    "quality",
    "mode_folder",
    "mode_label",
    "peptides",
    "total_labels",
    "uncertain_labels",
    "unspecific_labels",
    "specific_labels",
    "uncertain_share",
    "unspecific_share",
    "specific_share",
    "summary_path",
]]

if overview.empty:
    raise FileNotFoundError("No variant labeling files were found under the configured dataset roots.")

display(overview)
display(pd.DataFrame(pending_rows))

print(f"Found {source_rows} label rows across {len(overview)} available dataset/mode combinations.")
print(f"Pending first-quality dataset groups: {len(pending_rows)}")

def load_mode_class_frame(root, mode_specs):
    frame = None
    for mode_spec in mode_specs:
        summary_path = root / mode_spec["folder"] / mode_spec["summary_name"]
        if not summary_path.exists():
            continue

        mode_df = pd.read_csv(summary_path)
        required_columns = ["peptide", "n_spec_1", "n_spec_0", "n_spec_2"]
        missing_columns = [column for column in required_columns if column not in mode_df.columns]
        if missing_columns:
            raise ValueError(f"{summary_path} is missing columns: {missing_columns}")

        mode_df = mode_df[required_columns].rename(columns={
            "n_spec_1": f"{mode_spec['slug']}_specific",
            "n_spec_0": f"{mode_spec['slug']}_unspecific",
            "n_spec_2": f"{mode_spec['slug']}_uncertain",
        })
        frame = mode_df if frame is None else frame.merge(mode_df, on="peptide", how="outer")

    if frame is None:
        return pd.DataFrame()

    return frame.fillna(0).sort_values("peptide").reset_index(drop=True)


def plot_grouped_mode_stacks(ax, frame, title, mode_specs):
    if frame.empty:
        ax.axis("off")
        ax.text(0.5, 0.5, "Data pending", ha="center", va="center", fontsize=14, transform=ax.transAxes)
        ax.set_title(title)
        return 0

    peptides = frame["peptide"].tolist()
    positions = np.arange(len(peptides))
    bar_width = 0.18
    offsets = np.linspace(-1.5 * bar_width, 1.5 * bar_width, len(mode_specs))
    class_specs = [
        ("uncertain", "#D49A54"),
        ("unspecific", "#B6C7D6"),
        ("specific", "#2F6B8E"),
    ]

    max_height = 0
    for offset, mode_spec in zip(offsets, mode_specs):
        bottom = np.zeros(len(peptides))
        for class_name, color in class_specs:
            column_name = f"{mode_spec['slug']}_{class_name}"
            values = frame[column_name].to_numpy() if column_name in frame.columns else np.zeros(len(peptides))
            ax.bar(positions + offset, values, bar_width, bottom=bottom, color=color, edgecolor="black", linewidth=0.6)
            bottom = bottom + values
        max_height = max(max_height, float(bottom.max()) if len(bottom) else 0)

    ax.set_title(title)
    ax.set_xticks(positions)
    ax.set_xticklabels(peptides, rotation=45, ha="right")

    ax.tick_params(axis="x", which="major", pad=4, labelsize=8)

    ax.set_ylabel("Label count")
    ax.set_xlabel("Mode order within each peptide: combined | 3x | 3x1x | monotonic", fontsize=9, labelpad=14)
    ax.set_ylim(0, max_height * 1.05 if max_height else 1)
    return max_height


a3_q20_root = base / "P3408_LUCA-TCRA3/04_variant_labeling/07_01_2026_minlenght191_Q20_new"
a3_q30_root = base / "P3408_LUCA-TCRA3/04_variant_labeling/19_05_2026_minlenght191_Q30"
a3_mode_specs_q20 = [
    {"label": "combined", "slug": "combined", "folder": "combined", "summary_name": "variant_labeling_summary.csv"},
    {"label": "3x", "slug": "three_x", "folder": "single_positives_3x", "summary_name": "variant_labeling_summary.csv"},
    {"label": "3x1x", "slug": "three_x_one_x", "folder": "single_positives_3x1x", "summary_name": "variant_labeling_summary.csv"},
    {"label": "monotonic", "slug": "monotonic", "folder": "monotonic", "summary_name": "variant_labeling_summary.csv"},
]
a3_mode_specs_q30 = [
    {"label": "combined", "slug": "combined", "folder": "combined", "summary_name": "variant_labeling_summary.csv"},
    {"label": "3x", "slug": "three_x", "folder": "single_positive_3x", "summary_name": "variant_labeling_summary.csv"},
    {"label": "3x1x", "slug": "three_x_one_x", "folder": "single_positive_3x1x", "summary_name": "variant_labeling_summary.csv"},
    {"label": "monotonic", "slug": "monotonic", "folder": "monotonic", "summary_name": "variant_labeling_summary.csv"},
]

a3_q20_frame = load_mode_class_frame(a3_q20_root, a3_mode_specs_q20)
a3_q30_frame = load_mode_class_frame(a3_q30_root, a3_mode_specs_q30)
figure_a3, axes_a3 = plt.subplots(1, 2, figsize=(20, 7), sharey=True, constrained_layout=True)
a3_y_max_left = plot_grouped_mode_stacks(axes_a3[0], a3_q20_frame, "A3 - Q20", a3_mode_specs_q20)
a3_y_max_right = plot_grouped_mode_stacks(axes_a3[1], a3_q30_frame, "A3 - Q30", a3_mode_specs_q30)
shared_a3_max = max(a3_y_max_left, a3_y_max_right)
if shared_a3_max:
    axes_a3[0].set_ylim(0, shared_a3_max * 1.05)
    axes_a3[1].set_ylim(0, shared_a3_max * 1.05)
figure_a3.suptitle("A3: peptide-level mode comparison", fontsize=16)
figure_a3.legend(
    handles=[
        Patch(facecolor="#D49A54", edgecolor="black", label="uncertain"),
        Patch(facecolor="#B6C7D6", edgecolor="black", label="unspecific"),
        Patch(facecolor="#2F6B8E", edgecolor="black", label="specific"),
    ],
    title="Label class",
    loc="center left",
    bbox_to_anchor=(1.02, 0.5),
)
figure_a3.subplots_adjust(right=0.84, bottom=0.18)
plt.show()

# DMF5: detect available quality roots (Q20/Q30 or similar) and plot both subplots if present
dmf5_root = base / "P3481_LUCA-TCRDMF5/04_variant_labeling"
dmf5_mode_specs = [
    {"label": "combined", "slug": "combined", "folder": "combined", "summary_name": "variant_labeling_summary.csv"},
    {"label": "1x", "slug": "one_x", "folder": "single_positive_pos1x", "summary_name": "variant_labeling_summary.csv"},
    {"label": "2x", "slug": "two_x", "folder": "single_positive_pos2x", "summary_name": "variant_labeling_summary.csv"},
    {"label": "monotonic", "slug": "monotonic", "folder": "monotonic", "summary_name": "variant_labeling_summary.csv"},
]

dmf5_quality_roots = []
if dmf5_root.exists():
    for p in sorted(p for p in dmf5_root.iterdir() if p.is_dir()):
        if (p / "combined").exists():
            dmf5_quality_roots.append(p)

if not dmf5_quality_roots:
    dmf5_quality_roots = [dmf5_root]

if len(dmf5_quality_roots) >= 2:
    dmf5_q20_root, dmf5_q30_root = dmf5_quality_roots[:2]
else:
    dmf5_q20_root = dmf5_quality_roots[0]
    dmf5_q30_root = dmf5_quality_roots[0]

dmf5_q20_frame = load_mode_class_frame(dmf5_q20_root, dmf5_mode_specs)
dmf5_q30_frame = load_mode_class_frame(dmf5_q30_root, dmf5_mode_specs)

figure_dmf5, axes_dmf5 = plt.subplots(1, 2, figsize=(20, 7), sharey=True, constrained_layout=True)
dmf5_y_max_left = plot_grouped_mode_stacks(axes_dmf5[0], dmf5_q20_frame, f"DMF5 - {dmf5_q20_root.name}", dmf5_mode_specs)
dmf5_y_max_right = plot_grouped_mode_stacks(axes_dmf5[1], dmf5_q30_frame, f"DMF5 - {dmf5_q30_root.name}", dmf5_mode_specs)
shared_dmf5_max = max(dmf5_y_max_left, dmf5_y_max_right)
if shared_dmf5_max:
    axes_dmf5[0].set_ylim(0, shared_dmf5_max * 1.05)
    axes_dmf5[1].set_ylim(0, shared_dmf5_max * 1.05)

figure_dmf5.suptitle("DMF5: peptide-level mode comparison", fontsize=16)
figure_dmf5.legend(
    handles=[
        Patch(facecolor="#D49A54", edgecolor="black", label="uncertain"),
        Patch(facecolor="#B6C7D6", edgecolor="black", label="unspecific"),
        Patch(facecolor="#2F6B8E", edgecolor="black", label="specific"),
    ],
    title="Label class",
    loc="center left",
    bbox_to_anchor=(1.02, 0.5),
)
figure_dmf5.subplots_adjust(right=0.84, bottom=0.18)
plt.show()

TypeError: list indices must be integers or slices, not str